# Project 1 ANSC 4040 Model Formation and Training

This is a notebook for Project 1 that will be used for forming the initial model. The goal is to form a RandomForestClassifier to determine the AnimalId. The large limitation of this approach is that the model will NOT fucntion when new cows are added to the data set such as on a new farm. All cows in test set must be in the training set. 

In [1]:
import pandas as pd
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load Train and Validation Sets (starting with only 5 cows)

In [2]:
train_path = "train_data.csv"
val_path = "val_data.csv"

# Find the training-set cow IDs using a bounded-memory scan.
animal_ids = set()
for id_chunk in pd.read_csv(
    train_path,
    usecols=["AnimalId"],
    dtype={"AnimalId": "string"},
    chunksize=100_000,
):
    animal_ids.update(id_chunk["AnimalId"].dropna().unique())

if len(animal_ids) < 5:
    raise ValueError(f"Expected at least 5 cows in the training data; found {len(animal_ids)}.")

selected_animals = set(
    pd.Series(sorted(animal_ids)).sample(n=5, random_state=42).tolist()
)

# Read only model columns and rows for those five cows.
model_cols = [
    "AnimalId",
    "EventDate",
    "Avgmilkflow",
    "Flow30_60Session",
    "YieldFirst2Min_Session",
    "YieldSession",
    "DurationSession_sec",
    "milking",
]

def load_selected_cows(file_path):
    matching_chunks = []
    for chunk in pd.read_csv(
        file_path,
        usecols=model_cols,
        dtype={"AnimalId": "string"},
        chunksize=100_000,
    ):
        matching = chunk[chunk["AnimalId"].isin(selected_animals)]
        if not matching.empty:
            matching_chunks.append(matching)
    if not matching_chunks:
        raise ValueError(f"None of the selected cows were found in {file_path}.")
    return pd.concat(matching_chunks, ignore_index=True)

train = load_selected_cows(train_path)
val = load_selected_cows(val_path)

print(f"Selected cows: {sorted(selected_animals)}")
print(f"Training rows: {len(train):,}; validation rows: {len(val):,}")

# Split EventDate into numerical calendar features
for data in (train, val):
    event_date = pd.to_datetime(data["EventDate"])
    data["EventYear"] = event_date.dt.year
    data["EventMonth"] = event_date.dt.month
    data["EventDay"] = event_date.dt.day
    data["EventDayOfWeek"] = event_date.dt.dayofweek

Selected cows: ['-3.455482512856752e+18', '-4.1398554415718595e+18', '3.2600190913962895e+18', '4.162710639979213e+18', '4.5644797961052913e+18']
Training rows: 2,899; validation rows: 829


# Form Model: Select Target Variable and Initial Hyperparameters

In [3]:
# Target variable
target = "AnimalId"

# Define feature by including columns
feature_cols = ["Avgmilkflow", "EventYear", "EventMonth", "EventDay", "EventDayOfWeek", "Flow30_60Session", "YieldFirst2Min_Session", "YieldSession", "DurationSession_sec", "milking"]



X_train = train[feature_cols]
y_train = train[target]

X_val = val[feature_cols]
y_val = val[target]

rf_model = RandomForestClassifier(
        n_estimators=5,
        random_state=42
    )

# Fit model
rf_model.fit(X_train, y_train)

RandomForestClassifier(n_estimators=5, random_state=42)

In [4]:
# Evaluate on validation rows that were not used to fit the model.
y_val_pred = rf_model.predict(X_val)
val_accuracy = accuracy_score(y_val, y_val_pred)

print(f"Validation accuracy: {val_accuracy:.2%}")
print("\nClassification report:")
print(classification_report(y_val, y_val_pred, zero_division=0))

Validation accuracy: 46.44%

Classification report:
                         precision    recall  f1-score   support

 -3.455482512856752e+18       0.63      1.00      0.77        17
-4.1398554415718595e+18       0.96      0.19      0.32       118
 3.2600190913962895e+18       0.91      0.16      0.28       261
  4.162710639979213e+18       0.25      0.80      0.39       132
 4.5644797961052913e+18       0.63      0.65      0.64       301

               accuracy                           0.46       829
              macro avg       0.68      0.56      0.48       829
           weighted avg       0.71      0.46      0.44       829



# Hyperparameter tuning

This small grid searches a realistic set of Random Forest settings to find the best combination without making the tuning process too slow or too broad.

In [6]:
from sklearn.model_selection import GridSearchCV

# Hyperparameter tuning with GridSearchCV
# This explores a small grid of RandomForest settings and selects the combination
# with the best cross-validation accuracy.

param_grid = {
    "n_estimators": [5, 20, 50],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    scoring="accuracy",
    cv=3,
    n_jobs=-1,
    verbose=1,
)

grid_search.fit(X_train, y_train)

print(f"Best cross-validation accuracy: {grid_search.best_score_:.2%}")
print(f"Best parameters: {grid_search.best_params_}")

best_model = grid_search.best_estimator_
y_val_pred_best = best_model.predict(X_val)
best_val_accuracy = accuracy_score(y_val, y_val_pred_best)

print(f"Validation accuracy with tuned model: {best_val_accuracy:.2%}")
print("\nClassification report for tuned model:")
print(classification_report(y_val, y_val_pred_best, zero_division=0))

results_df = pd.DataFrame(grid_search.cv_results_).sort_values("mean_test_score", ascending=False)
print("\nTop GridSearchCV results:")
print(results_df[["params", "mean_test_score", "std_test_score"]].head(10).to_string(index=False))

Fitting 3 folds for each of 81 candidates, totalling 243 fits
Best cross-validation accuracy: 76.61%
Best parameters: {'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 20}
Validation accuracy with tuned model: 41.13%

Classification report for tuned model:
                         precision    recall  f1-score   support

 -3.455482512856752e+18       0.81      1.00      0.89        17
-4.1398554415718595e+18       0.68      0.21      0.32       118
 3.2600190913962895e+18       0.62      0.05      0.09       261
  4.162710639979213e+18       0.21      0.53      0.30       132
 4.5644797961052913e+18       0.52      0.72      0.60       301

               accuracy                           0.41       829
              macro avg       0.57      0.50      0.44       829
           weighted avg       0.53      0.41      0.36       829


Top GridSearchCV results:
                                                                              params  mean_test_sc